# Basic ADSim HDF5 Example

Configure an ADSimDetector with the shared area-detector factory, acquire
multiple frames, and inspect the HDF5 stream resource. This follows the
apstools `de_0_adsim_hdf5_basic` example.

The ADSimDetector IOC must already be running with prefix `ad:`. The IOC
writes below `/tmp`, while Python reads the same files below
`/tmp/docker_ioc/iocad/tmp`. These are different roots for the same
files. Adjust both paths for another IOC mount.

In [1]:
from pathlib import Path

from ophyd_async.core import UUIDFilenameProvider

from bits2606.devices.area_detector_factories import ad_creator_async
from example_path import ExampleDatePathProvider

In [2]:
IOC_PREFIX = "ad:"
IOC_MOUNT_PATH = Path("/tmp")
BLUESKY_MOUNT_PATH = Path("/local/ioc/iocad/tmp")
IOC_FILE_ROOT = IOC_MOUNT_PATH / "example"
CLIENT_FILE_ROOT = BLUESKY_MOUNT_PATH / "example"

path_provider = ExampleDatePathProvider(
    filename_provider=UUIDFilenameProvider(),
    directory_path=IOC_FILE_ROOT,
    directory_uri=(
        f"file://localhost/{CLIENT_FILE_ROOT.as_posix().lstrip('/')}"
    ),
    create_dir_depth=-3,
)
path_provider

In [3]:
print(f"{path_provider.directory_path=}")
print(f"{path_provider.directory_uri=}")
print(f"{path_provider.create_dir_depth=}")
print(f"{path_provider.filename_provider=}")

path_provider.directory_path=PosixPath('/tmp/example')
path_provider.directory_uri='file://localhost/local/ioc/iocad/tmp/example'
path_provider.create_dir_depth=-3
path_provider.filename_provider=<ophyd_async.core._path_providers.UUIDFilenameProvider object at 0x7f8c8b6ef380>


The detector is configured with the `image1:` plugin and the HDF5 writer.
The factory supplies the ADSim driver and acquisition logic.

In [4]:
adsimdet = ad_creator_async(
    prefix=IOC_PREFIX,
    name="adsimdet",
    plugin_list=["image"],
    writer_specs={"hdf": {"args": [path_provider]}},
)

print(adsimdet.name)
print(adsimdet.driver.name)

adsimdet
adsimdet-driver


Connect the factory-created device before using it in a Bluesky plan.
Top-level `await` is supported by Jupyter.

In [5]:
await adsimdet.connect(timeout=15)
print("connected")

connected


Create a temporary Tiled server for the life of this notebook. The server
uses temporary local storage and remains available while the `server`
variable is retained. `TiledWriter` receives each Bluesky document.

In [6]:
from bluesky import RunEngine
from bluesky import plans as bp
from bluesky_tiled_plugins import TiledWriter
from tiled.client import from_uri
from tiled.server import SimpleTiledServer

server = SimpleTiledServer()
cat = from_uri(server.uri)
tiled_writer = TiledWriter(cat, batch_size=1)
RE = RunEngine()
RE.subscribe(tiled_writer)

0

Acquire five frames in one run. The HDF5 writer uses Area Detector stream
mode, so the frames are stored in one HDF5 resource.

In [7]:
uids = RE(bp.count([adsimdet], num=5))
uids

('444c3b35-031e-4a6d-9905-5ff9b57ee766',)

In [8]:
assert len(uids) == 1
uids[0]

'444c3b35-031e-4a6d-9905-5ff9b57ee766'

Read the Bluesky run through Tiled. The temporary server and its data
remain available while the `server` object is retained by the notebook.
Use the public run and stream metadata. The HDF5 resource is external, so
inspect its client-visible URI directly rather than asking Tiled to fetch it.

In [9]:
run = cat[uids[0]]
run

<BlueskyRun v3.0 streams: {'primary'} scan_id=1 uid='444c3b35' 2026-10-23 20:19>

In [10]:
{
    "run_metadata": run.metadata,
    "primary_stream_metadata": run.primary.metadata,
}

{'run_metadata': {'start': {'detectors': ['adsimdet'],
           'hints': {'dimensions': [[['time'], 'primary']]},
           'num_intervals': 4,
           'num_points': 5,
           'plan_args': {'delay': 0.0,
                         'detectors': ['AreaDetector(name="adsimdet")'],
                         'num': 5},
           'plan_name': 'count',
           'plan_type': 'generator',
           'scan_id': 1,
           'time': 1792804750.3297293,
           'uid': '444c3b35-031e-4a6d-9905-5ff9b57ee766',
           'versions': {'bluesky': '1.15.1',
                        'event_model': '1.24.0',
                        'ophyd': '1.11.2',
                        'ophyd_async': '0.21.3'}},
 'stop': {'exit_status': 'success',
          'num_events': {'primary': 5},
          'reason': '',
          'run_start': '444c3b35-031e-4a6d-9905-5ff9b57ee766',
          'time': 1792804755.365543,
          'uid': '57b6b6d1-ee0c-4e82-a622-e4dda094d750'}},
 'primary_stream_metadata': {'configur

In [11]:
from pathlib import Path
from urllib.parse import urlparse

data_keys = run.primary.metadata["data_keys"]
hdf_key = next(key for key in data_keys if key.endswith("-hdf"))
source_uri = data_keys[hdf_key]["source"]
client_file = Path(urlparse(source_uri).path)
relative_file = client_file.relative_to(CLIENT_FILE_ROOT)
print(f"{hdf_key=}")
print(f"{IOC_FILE_ROOT=}")
print(f"{CLIENT_FILE_ROOT=}")
print(f"{source_uri=}")
print(f"{relative_file=}")
print(f"{client_file.exists()=} {client_file=}")

hdf_key='adsimdet-hdf'
IOC_FILE_ROOT=PosixPath('/tmp/example')
CLIENT_FILE_ROOT=PosixPath('/local/ioc/iocad/tmp/example')
source_uri='file://localhost/local/ioc/iocad/tmp/example/2026/10/23/940f2ec1-df36-4fb3-9774-1ff0528acac8_007.h5'
relative_file=PosixPath('2026/10/23/940f2ec1-df36-4fb3-9774-1ff0528acac8_007.h5')
client_file.exists()=True client_file=PosixPath('/local/ioc/iocad/tmp/example/2026/10/23/940f2ec1-df36-4fb3-9774-1ff0528acac8_007.h5')
